# Spotify Lakehouse — Data Inventory

A complete, honest inventory of every table this project can reach, with a real, redacted sample of each, so the breadth of the data — and its holes — are visible in one document. It covers **three sources** and what is built from them: the Spotify Web API (§1–§7), Spotify's Extended Streaming History export (§8), MusicBrainz (§9), and the warehouse (§10). API surfaces the warehouse already holds are read from it; the rest are called live, once.

Two constraints shape the API sections. **This is a development-mode app:** at most five Spotify accounts can authorize it, and the app owner needs Premium. And **it was registered after 2024-11-27**, so audio features, audio analysis, recommendations and related artists are closed to it (§6.1). Profile, user and owner fields, and identifier columns, are redacted throughout.

Everything starts from the project's notebook setup: it resolves the session, reads credentials from outside the repository, opens the warehouse connection and configures the plotting libraries. The run details and a coverage summary appear below.

In [ ]:
from spotify_lakehouse import data_inventory, inventory
from spotify_lakehouse.notebook import setup

ctx = setup(profile="marc")

The warehouse is read first. Each API surface not yet extracted is then called once, paced, under the notebook's own `spot_notebook` lock, so the scheduled poller is never made to skip.

In [ ]:
captures = inventory.collect(ctx)
inventory.emit_intro(ctx, captures)

## 1. Identity & Account

Who the authorized account is. The warehouse already stores `/me` responses, so this section reads the latest one rather than calling the API.

In [ ]:
inventory.emit_domain(captures, "1")

## 2. Library

Saved tracks, saved albums and followed artists. None is extracted yet, so each is called live once.

In [ ]:
inventory.emit_domain(captures, "2")

## 3. Playlists

The playlist inventory, then the contents of the first playlist. Not extracted yet; called live.

In [ ]:
inventory.emit_domain(captures, "3")

## 4. Listening

Recent plays come from the warehouse, where the poller lands them every 30 minutes. Spotify's own top artists and top tracks are called live for each of their three time ranges.

In [ ]:
inventory.emit_domain(captures, "4")

### 4.4 Local date versus UTC date

Every play is stored with a UTC timestamp, but a day of listening belongs to the listener's own calendar. This shows how many of the API's plays would move to a different date if they were grouped by UTC. The export's plays follow the same rule; they are left out here so the chart stays readable.

The chart counts the same API plays twice: once by their UTC date, once by their local date.

<!-- caption: API plays per calendar date, counted by UTC date and by the profile's home-timezone date -->

In [ ]:
inventory.emit_local_vs_utc(ctx)

### 4.5 The 50-item window is the whole API history

How much listening history the API path holds, when it starts, and where the export takes over.

In [ ]:
inventory.emit_api_window(ctx)

## 5. Catalog Enrichment

Catalog objects that describe what was played. Artists and tracks come from the warehouse (tracks since R-037 resolved export track ids one call each); albums, podcasts and search are called live.

In [ ]:
inventory.emit_domain(captures, "5")

## 6. What the API Cannot Give

The negative space of the API, with the round that measured each item and, now that other sources exist, what fills each gap — or a plain statement that nothing does. It is here so no future session spends an afternoon looking for something that is gone.

In [ ]:
inventory.emit_cannot_get(ctx, captures)

## 7. Coverage Summary

One row per API endpoint section: where its data came from, whether it was reachable, what was sampled, and which warehouse table it feeds.

In [ ]:
inventory.emit_coverage(captures)

## 8. The Extended Streaming History Export

Spotify's GDPR export: every play since the account was created — music, podcasts and audiobooks — with how long each lasted. Two tables: the records as loaded, and the typed staging view over them.

In [ ]:
data_inventory.emit_export(ctx)

## 9. MusicBrainz

The external source for genres. Spotify supplies none to this app, so each track's ISRC is looked up in MusicBrainz, then its primary artist's curated genres and open tags are fetched.

In [ ]:
data_inventory.emit_musicbrainz(ctx)

## 10. The Warehouse

The star as built from all three sources: the play fact, the content and artist dimensions, the genre bridge, and the allocation reconciliation that every genre figure must print first.

In [ ]:
data_inventory.emit_warehouse(ctx)

## 11. The Queryable Catalog

Everything the database holds, and how to start querying it. The list below is **introspected when this notebook runs** — from `information_schema` and `pg_catalog`, never typed out — so it is wrong only if the database is. Sections 8 to 10 explain what a handful of tables *hold*; this one answers what exists at all, what each object is for in its own words, and what a first query against it looks like.

In [ ]:
data_inventory.emit_catalog(ctx)

The warehouse connection is closed at the end of the run.

In [ ]:
ctx.close()